In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import json
import random
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix
)

SEEDS = [42, 123, 2024]

BASE_PROJECT = Path("/content/drive/MyDrive/New Jurnal Cross")

FEATURE_DIR = BASE_PROJECT / "processed_intra_features_e2v_plus_base"
OUT_DIR = BASE_PROJECT / "results_intra_emotion2vec_mlp_plus_base"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["emodb", "ravdess", "resd"]

LABELS = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
]

ID_TO_LABEL = {i: label for i, label in enumerate(LABELS)}
LABEL_TO_ID = {label: i for i, label in ID_TO_LABEL.items()}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("FEATURE_DIR:", FEATURE_DIR)
print("OUT_DIR:", OUT_DIR)

DEVICE: cuda
FEATURE_DIR: /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_e2v_plus_base
OUT_DIR: /content/drive/MyDrive/New Jurnal Cross/results_intra_emotion2vec_mlp_plus_base


In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_e2v_dataset(dataset_name, scale=True):
    ds_dir = FEATURE_DIR / dataset_name

    X_train = np.load(ds_dir / "X_e2v_train.npy").astype(np.float32)
    y_train = np.load(ds_dir / "y_train.npy").astype(np.int64)

    X_val = np.load(ds_dir / "X_e2v_val.npy").astype(np.float32)
    y_val = np.load(ds_dir / "y_val.npy").astype(np.int64)

    X_test = np.load(ds_dir / "X_e2v_test.npy").astype(np.float32)
    y_test = np.load(ds_dir / "y_test.npy").astype(np.int64)

    meta_train = pd.read_csv(ds_dir / "meta_train.csv")
    meta_val = pd.read_csv(ds_dir / "meta_val.csv")
    meta_test = pd.read_csv(ds_dir / "meta_test.csv")

    scaler = None
    if scale:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train).astype(np.float32)
        X_val = scaler.transform(X_val).astype(np.float32)
        X_test = scaler.transform(X_test).astype(np.float32)

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "meta_train": meta_train,
        "meta_val": meta_val,
        "meta_test": meta_test,
        "scaler": scaler,
    }


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "uar": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def make_report_df(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        target_names=LABELS,
        labels=list(range(len(LABELS))),
        zero_division=0,
        output_dict=True
    )
    return pd.DataFrame(report).transpose()


def save_confusion_matrix_csv(cm, out_path):
    df_cm = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    df_cm.to_csv(out_path, index=True)


def make_loader(X, y, batch_size=32, shuffle=False):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)

    dataset = TensorDataset(X_tensor, y_tensor)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False
    )


def compute_class_weights(y_train, n_classes=6):
    counts = np.bincount(y_train, minlength=n_classes).astype(np.float32)
    weights = counts.sum() / (n_classes * counts)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)

In [4]:
class Emotion2VecMLP(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim=256,
        num_classes=6,
        dropout=0.30
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [5]:
def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_targets = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_targets.extend(y_batch.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(np.array(all_targets), np.array(all_preds))

    return avg_loss, metrics, np.array(all_targets), np.array(all_preds)


@torch.no_grad()
def predict_model(model, loader):
    model.eval()

    all_preds = []
    all_targets = []
    all_probs = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)

        logits = model(X_batch)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_targets.extend(y_batch.numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

    return (
        np.array(all_targets),
        np.array(all_preds),
        np.array(all_probs)
    )

In [6]:
def train_eval_e2v_mlp(
    dataset_name,
    seed,
    batch_size=32,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=150,
    patience=20,
    scale=True
):
    set_seed(seed)

    data = load_e2v_dataset(dataset_name, scale=scale)

    X_train = data["X_train"]
    y_train = data["y_train"]

    X_val = data["X_val"]
    y_val = data["y_val"]

    X_test = data["X_test"]
    y_test = data["y_test"]

    input_dim = X_train.shape[1]
    num_classes = len(LABELS)

    train_loader = make_loader(X_train, y_train, batch_size=batch_size, shuffle=True)
    val_loader = make_loader(X_val, y_val, batch_size=batch_size, shuffle=False)
    test_loader = make_loader(X_test, y_test, batch_size=batch_size, shuffle=False)

    model = Emotion2VecMLP(
        input_dim=input_dim,
        hidden_dim=256,
        num_classes=num_classes,
        dropout=0.30
    ).to(DEVICE)

    class_weights = compute_class_weights(y_train, n_classes=num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5
    )

    best_val_macro_f1 = -1.0
    best_epoch = -1
    best_state = None
    no_improve = 0

    history = []

    for epoch in range(1, max_epochs + 1):
        train_loss, train_metrics, _, _ = run_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer
        )

        val_loss, val_metrics, _, _ = run_one_epoch(
            model,
            val_loader,
            criterion,
            optimizer=None
        )

        scheduler.step(val_metrics["macro_f1"])

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "lr": optimizer.param_groups[0]["lr"]
        }
        history.append(row)

        current = val_metrics["macro_f1"]

        if current > best_val_macro_f1:
            best_val_macro_f1 = current
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"[{dataset_name} | seed={seed}] "
                f"Epoch {epoch:03d} | "
                f"train_loss={train_loss:.4f} | "
                f"val_loss={val_loss:.4f} | "
                f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
                f"best={best_val_macro_f1:.4f}"
            )

        if no_improve >= patience:
            print(
                f"[{dataset_name} | seed={seed}] Early stopping at epoch {epoch}. "
                f"Best epoch: {best_epoch}, best val macro-F1: {best_val_macro_f1:.4f}"
            )
            break

    model.load_state_dict(best_state)

    y_val_true, y_val_pred, y_val_prob = predict_model(model, val_loader)
    y_test_true, y_test_pred, y_test_prob = predict_model(model, test_loader)

    val_metrics = compute_metrics(y_val_true, y_val_pred)
    test_metrics = compute_metrics(y_test_true, y_test_pred)

    run_dir = OUT_DIR / dataset_name / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), run_dir / "e2v_mlp_model.pt")

    if data["scaler"] is not None:
        import pickle
        with open(run_dir / "scaler_e2v.pkl", "wb") as f:
            pickle.dump(data["scaler"], f)

    with open(run_dir / "training_config.json", "w") as f:
        json.dump({
            "dataset": dataset_name,
            "seed": seed,
            "batch_size": batch_size,
            "lr": lr,
            "weight_decay": weight_decay,
            "max_epochs": max_epochs,
            "patience": patience,
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_macro_f1,
            "input_dim": input_dim,
            "num_classes": num_classes,
            "model": "Emotion2VecMLP",
            "class_weighted_loss": True,
            "scale": scale
        }, f, indent=2)

    pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)

    pd.DataFrame([{
        "dataset": dataset_name,
        "seed": seed,
        "split": "val",
        "best_epoch": best_epoch,
        **val_metrics
    }]).to_csv(run_dir / "val_metrics.csv", index=False)

    pd.DataFrame([{
        "dataset": dataset_name,
        "seed": seed,
        "split": "test",
        "best_epoch": best_epoch,
        **test_metrics
    }]).to_csv(run_dir / "test_metrics.csv", index=False)

    make_report_df(y_val_true, y_val_pred).to_csv(run_dir / "val_classification_report.csv")
    make_report_df(y_test_true, y_test_pred).to_csv(run_dir / "test_classification_report.csv")

    cm_val = confusion_matrix(y_val_true, y_val_pred, labels=list(range(len(LABELS))))
    cm_test = confusion_matrix(y_test_true, y_test_pred, labels=list(range(len(LABELS))))

    save_confusion_matrix_csv(cm_val, run_dir / "val_confusion_matrix.csv")
    save_confusion_matrix_csv(cm_test, run_dir / "test_confusion_matrix.csv")

    pred_val_df = data["meta_val"].copy()
    pred_val_df["y_true"] = y_val_true
    pred_val_df["y_pred"] = y_val_pred
    pred_val_df["true_label"] = [ID_TO_LABEL[i] for i in y_val_true]
    pred_val_df["pred_label"] = [ID_TO_LABEL[i] for i in y_val_pred]
    for i, label in enumerate(LABELS):
        pred_val_df[f"prob_{label}"] = y_val_prob[:, i]
    pred_val_df.to_csv(run_dir / "val_predictions.csv", index=False)

    pred_test_df = data["meta_test"].copy()
    pred_test_df["y_true"] = y_test_true
    pred_test_df["y_pred"] = y_test_pred
    pred_test_df["true_label"] = [ID_TO_LABEL[i] for i in y_test_true]
    pred_test_df["pred_label"] = [ID_TO_LABEL[i] for i in y_test_pred]
    for i, label in enumerate(LABELS):
        pred_test_df[f"prob_{label}"] = y_test_prob[:, i]
    pred_test_df.to_csv(run_dir / "test_predictions.csv", index=False)

    row_val = {
        "dataset": dataset_name,
        "seed": seed,
        "split": "val",
        "best_epoch": best_epoch,
        **val_metrics
    }

    row_test = {
        "dataset": dataset_name,
        "seed": seed,
        "split": "test",
        "best_epoch": best_epoch,
        **test_metrics
    }

    return row_val, row_test

In [7]:
all_rows = []

for dataset_name in DATASETS:
    print("=" * 100)
    print(f"DATASET: {dataset_name.upper()}")
    print("=" * 100)

    for seed in SEEDS:
        print(f"\nTraining emotion2vec-only MLP | dataset={dataset_name} | seed={seed}")

        row_val, row_test = train_eval_e2v_mlp(
            dataset_name=dataset_name,
            seed=seed,
            batch_size=32,
            lr=1e-3,
            weight_decay=1e-4,
            max_epochs=150,
            patience=20,
            scale=True
        )

        all_rows.append(row_val)
        all_rows.append(row_test)

        print("VAL :", {k: round(v, 4) for k, v in row_val.items() if isinstance(v, float)})
        print("TEST:", {k: round(v, 4) for k, v in row_test.items() if isinstance(v, float)})

results = pd.DataFrame(all_rows)
results.to_csv(OUT_DIR / "all_seed_results.csv", index=False)

display(results)
print("Saved:", OUT_DIR / "all_seed_results.csv")

DATASET: EMODB

Training emotion2vec-only MLP | dataset=emodb | seed=42
[emodb | seed=42] Epoch 001 | train_loss=0.7983 | val_loss=0.8813 | val_macro_f1=0.7079 | best=0.7079
[emodb | seed=42] Epoch 010 | train_loss=0.2952 | val_loss=0.9775 | val_macro_f1=0.7078 | best=0.7424
[emodb | seed=42] Epoch 020 | train_loss=0.2161 | val_loss=1.0147 | val_macro_f1=0.7078 | best=0.7558
[emodb | seed=42] Epoch 030 | train_loss=0.1839 | val_loss=1.0720 | val_macro_f1=0.7078 | best=0.7558
[emodb | seed=42] Early stopping at epoch 38. Best epoch: 18, best val macro-F1: 0.7558
VAL : {'accuracy': 0.7606, 'macro_f1': 0.7558, 'weighted_f1': 0.7531, 'uar': 0.769}
TEST: {'accuracy': 0.9262, 'macro_f1': 0.927, 'weighted_f1': 0.9259, 'uar': 0.9271}

Training emotion2vec-only MLP | dataset=emodb | seed=123
[emodb | seed=123] Epoch 001 | train_loss=0.7561 | val_loss=0.9098 | val_macro_f1=0.7396 | best=0.7396
[emodb | seed=123] Epoch 010 | train_loss=0.2916 | val_loss=1.0260 | val_macro_f1=0.7078 | best=0.7584


,dataset,seed,split,best_epoch,accuracy,macro_f1,weighted_f1,uar
0,emodb,42,val,18,0.760563,0.755824,0.753121,0.769037
1,emodb,42,test,18,0.926174,0.927021,0.925860,0.927109
2,emodb,123,val,4,0.760563,0.758391,0.754233,0.770105
3,emodb,123,test,4,0.899329,0.899887,0.899163,0.896944
4,emodb,2024,val,3,0.746479,0.737796,0.739250,0.754079
5,emodb,2024,test,3,0.919463,0.919946,0.918963,0.918762
6,ravdess,42,val,32,0.937500,0.932286,0.938415,0.942708
7,ravdess,42,test,32,0.977273,0.973412,0.977055,0.968750
8,ravdess,123,val,18,0.931818,0.925821,0.933431,0.937500
9,ravdess,123,test,18,0.977273,0.973412,0.977055,0.968750


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_emotion2vec_mlp_plus_base/all_seed_results.csv


In [8]:
metrics = ["accuracy", "macro_f1", "weighted_f1", "uar"]

summary_rows = []

for dataset_name in DATASETS:
    for split in ["val", "test"]:
        sub = results[
            (results["dataset"] == dataset_name) &
            (results["split"] == split)
        ]

        row = {
            "dataset": dataset_name,
            "split": split,
            "n_seeds": len(sub),
            "best_epoch_mean": sub["best_epoch"].mean(),
            "best_epoch_std": sub["best_epoch"].std(ddof=1),
        }

        for metric in metrics:
            row[f"{metric}_mean"] = sub[metric].mean()
            row[f"{metric}_std"] = sub[metric].std(ddof=1)

        summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / "summary_mean_std.csv", index=False)

display(summary)
print("Saved:", OUT_DIR / "summary_mean_std.csv")

,dataset,split,n_seeds,best_epoch_mean,best_epoch_std,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,uar_mean,uar_std
0,emodb,val,3,8.333333,8.386497,0.755869,8.131694e-03,0.750670,0.011223,0.748868,0.008348,0.764407,0.008960
1,emodb,test,3,8.333333,8.386497,0.914989,1.397091e-02,0.915618,0.014075,0.914662,0.013859,0.914272,0.015576
2,ravdess,val,3,18.333333,13.503086,0.929924,8.679121e-03,0.924036,0.009272,0.931449,0.008141,0.935764,0.007956
3,ravdess,test,3,18.333333,13.503086,0.977273,1.359740e-16,0.973412,0.000000,0.977055,0.000000,0.968750,0.000000
4,resd,val,3,13.000000,6.082763,0.661290,0.000000e+00,0.653999,0.001454,0.648882,0.001954,0.666998,0.002562
5,resd,test,3,13.000000,6.082763,0.565947,1.497601e-02,0.546025,0.017867,0.566678,0.015246,0.567849,0.019836


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_emotion2vec_mlp_plus_base/summary_mean_std.csv


In [9]:
def mean_std_str(mean, std, scale=100):
    return f"{mean * scale:.2f} ± {std * scale:.2f}"


paper_rows = []

for dataset_name in DATASETS:
    sub = summary[
        (summary["dataset"] == dataset_name) &
        (summary["split"] == "test")
    ].iloc[0]

    paper_rows.append({
        "Dataset": dataset_name.upper(),
        "Accuracy": mean_std_str(sub["accuracy_mean"], sub["accuracy_std"]),
        "UAR": mean_std_str(sub["uar_mean"], sub["uar_std"]),
        "Macro-F1": mean_std_str(sub["macro_f1_mean"], sub["macro_f1_std"]),
        "Weighted-F1": mean_std_str(sub["weighted_f1_mean"], sub["weighted_f1_std"]),
        "Best Epoch": f"{sub['best_epoch_mean']:.1f} ± {sub['best_epoch_std']:.1f}",
    })

paper_table = pd.DataFrame(paper_rows)
paper_table.to_csv(OUT_DIR / "paper_table_test_mean_std.csv", index=False)

display(paper_table)
print("Saved:", OUT_DIR / "paper_table_test_mean_std.csv")

,Dataset,Accuracy,UAR,Macro-F1,Weighted-F1,Best Epoch
0,EMODB,91.50 ± 1.40,91.43 ± 1.56,91.56 ± 1.41,91.47 ± 1.39,8.3 ± 8.4
1,RAVDESS,97.73 ± 0.00,96.88 ± 0.00,97.34 ± 0.00,97.71 ± 0.00,18.3 ± 13.5
2,RESD,56.59 ± 1.50,56.78 ± 1.98,54.60 ± 1.79,56.67 ± 1.52,13.0 ± 6.1


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_emotion2vec_mlp_plus_base/paper_table_test_mean_std.csv


In [10]:
SVM_DIR = BASE_PROJECT / "results_intra_handcrafted_svm"
HC_MLP_DIR = BASE_PROJECT / "results_intra_handcrafted_mlp"
E2V_DIR = BASE_PROJECT / "results_intra_emotion2vec_mlp_plus_base"

svm_summary = pd.read_csv(SVM_DIR / "summary_mean_std.csv")
hc_mlp_summary = pd.read_csv(HC_MLP_DIR / "summary_mean_std.csv")
e2v_summary = pd.read_csv(E2V_DIR / "summary_mean_std.csv")

svm_test = svm_summary[svm_summary["split"] == "test"].copy()
hc_mlp_test = hc_mlp_summary[hc_mlp_summary["split"] == "test"].copy()
e2v_test = e2v_summary[e2v_summary["split"] == "test"].copy()

svm_test["model"] = "Handcrafted SVM-RBF"
hc_mlp_test["model"] = "Handcrafted MLP"
e2v_test["model"] = "emotion2vec MLP"

compare = pd.concat([svm_test, hc_mlp_test, e2v_test], ignore_index=True)

cols = [
    "model",
    "dataset",
    "accuracy_mean",
    "accuracy_std",
    "uar_mean",
    "uar_std",
    "macro_f1_mean",
    "macro_f1_std",
    "weighted_f1_mean",
    "weighted_f1_std",
]

compare = compare[cols]
compare.to_csv(E2V_DIR / "compare_handcrafted_vs_e2v_test.csv", index=False)

display(compare)
print("Saved:", E2V_DIR / "compare_handcrafted_vs_e2v_test.csv")

,model,dataset,accuracy_mean,accuracy_std,uar_mean,uar_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std
0,Handcrafted SVM-RBF,emodb,0.765101,0.000000e+00,0.771862,0.000000,0.770845,0.000000,0.766550,0.000000
1,Handcrafted SVM-RBF,ravdess,0.585227,0.000000e+00,0.572917,0.000000,0.562632,0.000000,0.575213,0.000000
2,Handcrafted SVM-RBF,resd,0.230216,0.000000e+00,0.216041,0.000000,0.177384,0.000000,0.201610,0.000000
3,Handcrafted MLP,emodb,0.722595,5.125924e-02,0.722497,0.057131,0.724663,0.053999,0.723408,0.052484
4,Handcrafted MLP,ravdess,0.564394,3.280399e-03,0.576389,0.003007,0.556696,0.007691,0.556044,0.004402
5,Handcrafted MLP,resd,0.213429,3.244065e-02,0.205268,0.026294,0.208880,0.041376,0.228542,0.040862
6,emotion2vec MLP,emodb,0.914989,1.397091e-02,0.914272,0.015576,0.915618,0.014075,0.914662,0.013859
7,emotion2vec MLP,ravdess,0.977273,1.359740e-16,0.968750,0.000000,0.973412,0.000000,0.977055,0.000000
8,emotion2vec MLP,resd,0.565947,1.497601e-02,0.567849,0.019836,0.546025,0.017867,0.566678,0.015246


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_emotion2vec_mlp_plus_base/compare_handcrafted_vs_e2v_test.csv
